# LightGBM vs Embedding - precision / recall / F1

Both models are scored on the **same pairs with the same labels**, so the
numbers are directly comparable:

| | LightGBM | Embedding |
|---|---|---|
| input | the 19 precomputed runtime features | offer text vs master text |
| output | `predict_proba` | cosine similarity |
| decision | score >= threshold -> predicted match | score >= threshold -> predicted match |

Each model gets metrics twice: at its **shipped/default threshold** (what it
would actually do in production) and at its **best-F1 threshold** (its ceiling,
tuned on this very data - optimistic, quoted only for comparison).

### Upload these three files
```
models/alkabeer_sku_matcher_v1.joblib
data/processed/test_split.parquet
Product_Master.xlsx
```
Optional: `validation_split.parquet` (to pick the threshold honestly - see §9).

## 1. Setup

In [ ]:
!pip -q install sentence-transformers rapidfuzz lightgbm joblib openpyxl pyarrow scikit-learn 2>/dev/null

import json, os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

from sklearn.metrics import (precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score,
                             precision_recall_curve, confusion_matrix,
                             classification_report)
try:
    import torch; DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
except ImportError:
    DEVICE = 'cpu'
print('device:', DEVICE)

## 2. Paths

In [ ]:
NEEDS_SRC = False

try:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/promostrater'
except Exception:
    BASE = os.environ.get('PROMOSTRATER_DIR', '.')
print('BASE =', BASE)

import glob
def find_file(fname):
    """Find a file ANYWHERE under BASE - the folder layout does not matter."""
    hits = [p for p in glob.glob(f'{BASE}/**/{fname}', recursive=True)
            if os.path.isfile(p)]
    hits.sort(key=len)
    return hits[0] if hits else None

def find_src():
    hits = glob.glob(f'{BASE}/**/sku_mapping/__init__.py', recursive=True)
    hits.sort(key=len)
    return os.path.dirname(os.path.dirname(hits[0])) if hits else None

MODEL_PATH  = find_file('alkabeer_sku_matcher_v1.joblib')
MASTER_PATH = find_file('Product_Master.xlsx')
TEST_PATH   = find_file('test_split.parquet')
VAL_PATH    = find_file('validation_split.parquet')
ALL_PATH    = find_file('training_features.parquet')
SRC_DIR     = find_src()

required = {'LightGBM model': MODEL_PATH, 'Product Master': MASTER_PATH,
            'test split': TEST_PATH}
if NEEDS_SRC:
    required['src package'] = SRC_DIR
missing = [k for k, v in required.items() if not v]
for k, v in required.items():
    print(f"{'OK     ' if v else 'MISSING'}  {k:15s} {v or '-- not found'}")
for k, v in (('validation split', VAL_PATH), ('all gold pairs', ALL_PATH)):
    print(f"{'OK     ' if v else 'absent '}  {k:15s} {v or '(optional)'}")

if missing:
    print(f'\n--- what IS present under {BASE} ---')
    seen = sorted(glob.glob(f'{BASE}/**/*', recursive=True))
    for p in seen[:50]:
        print('   ', p)
    if not seen:
        print('    (nothing - is the folder name exactly "promostrater"?)')
    raise FileNotFoundError('Could not find: ' + ', '.join(missing))


## 3. Load the evaluation set

In [ ]:
import joblib
bundle = joblib.load(MODEL_PATH)
CLF       = bundle['model']
FEATS     = list(bundle['feature_columns'])
AUTO_T    = float(bundle.get('auto_match_threshold', 0.5))
REVIEW_T  = float(bundle.get('manual_review_threshold', 0.5))
print('model version    :', bundle.get('model_version'))
print('auto threshold   :', AUTO_T)
print('review threshold :', REVIEW_T)
print('features         :', len(FEATS))

test = pd.read_parquet(TEST_PATH)
test['master_itemcode'] = test['master_itemcode'].astype(str)
y = test['pair_label'].astype(int).to_numpy()
print(f'\ntest pairs: {len(test)}   positives: {y.sum()}   negatives: {(1-y).sum()}'
      f'   positive rate: {y.mean():.1%}')
missing = [c for c in FEATS if c not in test.columns]
assert not missing, f'test split missing model features: {missing}'
print('all 19 model features present in the test split')

## 4. Model A - LightGBM

Uses the precomputed runtime features exactly as production does.

In [ ]:
X = test[FEATS].astype(float).fillna(-1)
if hasattr(CLF, 'predict_proba'):
    score_ml = CLF.predict_proba(X)[:, 1]
else:
    score_ml = np.asarray(CLF.predict(X), dtype=float)
print('LightGBM scores  min %.3f  median %.3f  max %.3f'
      % (score_ml.min(), np.median(score_ml), score_ml.max()))
print('mean score | positives %.3f   negatives %.3f'
      % (score_ml[y == 1].mean(), score_ml[y == 0].mean()))

## 5. Model B - Embedding

Cosine similarity between the offer text and the master SKU text, on the very
same pairs. Master text mirrors the runtime construction
(`Itemname + Item-Cat-4 + Item Description`).

In [ ]:
from sentence_transformers import SentenceTransformer

EMB_MODEL = 'BAAI/bge-small-en-v1.5'
emb = SentenceTransformer(EMB_MODEL, device=DEVICE)

mst = pd.read_excel(MASTER_PATH)
for c in ['Itemname', 'Item-Cat-4', 'Item Description']:
    mst[c] = mst[c].fillna('')
mst['Itemcode'] = (mst['Itemcode'].fillna('').astype(str)
                   .str.replace(r'\.0$', '', regex=True).str.strip())
mst = mst.drop_duplicates('Itemcode').reset_index(drop=True)
mst['master_text'] = (mst['Itemname'] + ' ' + mst['Item-Cat-4'] + ' '
                      + mst['Item Description']).str.replace(r'\s+', ' ', regex=True).str.strip()
code2text = dict(zip(mst['Itemcode'], mst['master_text']))

unknown = sorted(set(test['master_itemcode']) - set(code2text))
print(f'test SKUs absent from Product Master: {len(unknown)}', unknown[:5])

offer_txt  = test['offer_text'].astype(str).tolist()
master_txt = [code2text.get(c, '') for c in test['master_itemcode']]

def l2(a):
    return a / np.clip(np.linalg.norm(a, axis=1, keepdims=True), 1e-12, None)

# encode unique strings only, then map back
uo = sorted(set(offer_txt)); um = sorted(set(master_txt))
vo = l2(emb.encode(uo, batch_size=64, convert_to_numpy=True, show_progress_bar=True))
vm = l2(emb.encode(um, batch_size=64, convert_to_numpy=True, show_progress_bar=True))
oi = {t: i for i, t in enumerate(uo)}; mi_ = {t: i for i, t in enumerate(um)}
score_emb = np.array([float(vo[oi[o]] @ vm[mi_[m]])
                      for o, m in zip(offer_txt, master_txt)])
print('\nEmbedding cosine  min %.3f  median %.3f  max %.3f'
      % (score_emb.min(), np.median(score_emb), score_emb.max()))
print('mean cosine | positives %.3f   negatives %.3f'
      % (score_emb[y == 1].mean(), score_emb[y == 0].mean()))

## 6. Precision / recall / F1

In [ ]:
def metrics_at(y_true, score, t):
    pred = (score >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        'threshold': round(float(t), 4),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall':    recall_score(y_true, pred, zero_division=0),
        'f1':        f1_score(y_true, pred, zero_division=0),
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
        'predicted_positive': int(pred.sum()),
    }

def best_f1(y_true, score):
    p, r, th = precision_recall_curve(y_true, score)
    f = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)
    k = int(np.nanargmax(f[:-1])) if len(th) else 0
    return float(th[k]) if len(th) else 0.5

rows = []
for name, sc, ship in [('LightGBM', score_ml, AUTO_T),
                       ('Embedding', score_emb, None)]:
    bt = best_f1(y, sc)
    if ship is not None:
        rows.append({'model': name, 'operating_point': f'shipped ({ship})',
                     **metrics_at(y, sc, ship)})
    rows.append({'model': name, 'operating_point': 'best-F1 (tuned on test)',
                 **metrics_at(y, sc, bt)})
    rows.append({'model': name, 'operating_point': 'AP / ROC-AUC',
                 'threshold': np.nan,
                 'precision': average_precision_score(y, sc),
                 'recall': np.nan, 'f1': np.nan,
                 'TP': np.nan, 'FP': np.nan, 'FN': np.nan, 'TN': np.nan,
                 'predicted_positive': roc_auc_score(y, sc)})

res = pd.DataFrame(rows)
show = res[res.operating_point != 'AP / ROC-AUC'].copy()
for c in ('precision', 'recall', 'f1'):
    show[c] = show[c].map(lambda v: f'{v:.3f}')
print('=' * 78)
print('PRECISION / RECALL / F1  -  identical pairs, identical labels')
print('=' * 78)
print(show[['model', 'operating_point', 'threshold', 'precision', 'recall', 'f1',
            'TP', 'FP', 'FN', 'TN']].to_string(index=False))

rank = res[res.operating_point == 'AP / ROC-AUC']
print('\nthreshold-free ranking quality')
for _, r in rank.iterrows():
    print(f"  {r['model']:10s} average precision {r['precision']:.3f}   "
          f"ROC-AUC {r['predicted_positive']:.3f}")
print(f"\nbaseline (label everything positive): precision {y.mean():.3f}  recall 1.000  "
      f"f1 {2*y.mean()/(1+y.mean()):.3f}")

## 7. Per-class report and confusion matrices

In [ ]:
for name, sc, t in [('LightGBM', score_ml, AUTO_T),
                    ('Embedding', score_emb, best_f1(y, score_emb))]:
    print('=' * 60); print(f'{name}  @ threshold {t:.4f}'); print('=' * 60)
    pred = (sc >= t).astype(int)
    print(classification_report(y, pred, target_names=['no-match', 'match'],
                                zero_division=0, digits=3))
    cm = confusion_matrix(y, pred, labels=[0, 1])
    print(pd.DataFrame(cm, index=['actual no-match', 'actual match'],
                       columns=['pred no-match', 'pred match']).to_string(), '\n')

## 8. Precision-recall curves

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for name, sc in [('LightGBM', score_ml), ('Embedding', score_emb)]:
    p, r, _ = precision_recall_curve(y, sc)
    ax[0].plot(r, p, label=f'{name} (AP={average_precision_score(y, sc):.3f})')
ax[0].axhline(y.mean(), ls='--', c='grey', lw=1, label=f'chance ({y.mean():.2f})')
ax[0].set_xlabel('recall'); ax[0].set_ylabel('precision')
ax[0].set_title('Precision-Recall'); ax[0].legend(); ax[0].grid(alpha=.3)

for name, sc in [('LightGBM', score_ml), ('Embedding', score_emb)]:
    ts = np.linspace(sc.min(), sc.max(), 200)
    ax[1].plot(ts, [f1_score(y, (sc >= t).astype(int), zero_division=0) for t in ts],
               label=name)
ax[1].set_xlabel('threshold'); ax[1].set_ylabel('F1')
ax[1].set_title('F1 vs threshold'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 9. Honest threshold (tuned on validation, measured on test)

Section 6's best-F1 row is optimistic because the threshold was chosen on the
same rows it is scored on. If `validation_split.parquet` is present, the
threshold is chosen there and applied here - that is the number to quote.

In [ ]:
if os.path.exists(VAL_PATH):
    val = pd.read_parquet(VAL_PATH)
    val['master_itemcode'] = val['master_itemcode'].astype(str)
    yv = val['pair_label'].astype(int).to_numpy()
    sv_ml = (CLF.predict_proba(val[FEATS].astype(float).fillna(-1))[:, 1]
             if hasattr(CLF, 'predict_proba')
             else np.asarray(CLF.predict(val[FEATS].astype(float).fillna(-1)), float))
    vo_t = val['offer_text'].astype(str).tolist()
    vm_t = [code2text.get(c, '') for c in val['master_itemcode']]
    a = l2(emb.encode(vo_t, batch_size=64, convert_to_numpy=True, show_progress_bar=False))
    b = l2(emb.encode(vm_t, batch_size=64, convert_to_numpy=True, show_progress_bar=False))
    sv_emb = np.sum(a * b, axis=1)

    out = []
    for name, sval, stest in [('LightGBM', sv_ml, score_ml),
                              ('Embedding', sv_emb, score_emb)]:
        t = best_f1(yv, sval)
        out.append({'model': name, 'threshold_from_validation': round(t, 4),
                    'val_f1': round(f1_score(yv, (sval >= t).astype(int),
                                             zero_division=0), 3),
                    **{k: (round(v, 3) if isinstance(v, float) else v)
                       for k, v in metrics_at(y, stest, t).items()
                       if k in ('precision', 'recall', 'f1', 'TP', 'FP', 'FN', 'TN')}})
    hon = pd.DataFrame(out)
    print('threshold picked on VALIDATION, metrics measured on TEST')
    print(hon.to_string(index=False))
    print('\nA large val_f1 -> f1 drop means the threshold does not generalize.')
else:
    print('validation_split.parquet not uploaded - skipping. Section 6 best-F1 '
          'numbers remain optimistic.')

## 10. Where each model fails, and where they disagree

In [ ]:
t_ml, t_em = AUTO_T, best_f1(y, score_emb)
p_ml, p_em = (score_ml >= t_ml).astype(int), (score_emb >= t_em).astype(int)
det = test[['offer_text', 'master_itemcode', 'pair_label', 'label_provenance']].copy()
det['ml_score'] = score_ml.round(3); det['ml_pred'] = p_ml
det['emb_score'] = score_emb.round(3); det['emb_pred'] = p_em
det['ml_correct'] = det.ml_pred == det.pair_label
det['emb_correct'] = det.emb_pred == det.pair_label

print('agreement between the two models: %.1f%%' % ((p_ml == p_em).mean() * 100))
print('both correct        :', int((det.ml_correct & det.emb_correct).sum()))
print('only LightGBM right :', int((det.ml_correct & ~det.emb_correct).sum()))
print('only Embedding right:', int((~det.ml_correct & det.emb_correct).sum()))
print('both wrong          :', int((~det.ml_correct & ~det.emb_correct).sum()))

print('\naccuracy by label provenance:')
print(det.groupby('label_provenance')[['ml_correct', 'emb_correct']]
      .agg(['mean', 'count']).round(3).to_string())

print('\nmissed matches - true positives BOTH models rejected:')
miss = det[(det.pair_label == 1) & (~det.ml_correct) & (~det.emb_correct)]
print(miss[['offer_text', 'master_itemcode', 'ml_score', 'emb_score']]
      .head(10).to_string(index=False))

print('\nfalse matches - negatives BOTH models accepted:')
fp = det[(det.pair_label == 0) & (~det.ml_correct) & (~det.emb_correct)]
print(fp[['offer_text', 'master_itemcode', 'ml_score', 'emb_score']]
      .head(10).to_string(index=False))

det.to_csv('per_pair_predictions.csv', index=False)
show.to_csv('metrics_summary.csv', index=False)
print('\nwrote per_pair_predictions.csv and metrics_summary.csv')

## 11. How to read this

- **Precision** = of the pairs a model called a match, how many really were.
  This is the number that governs whether auto-accept is safe.
- **Recall** = of the real matches, how many the model found.
- **F1** = their harmonic mean; use it to compare the two models at one glance.

Two caveats to carry into any decision made from these numbers:

1. **The test labels are not human-verified.** They come from the same
   auto-labelled pool audited earlier, where a share of positives were wrong.
   Both models are therefore scored against imperfect ground truth, and the
   `label_provenance` breakdown in §10 is the place to look for that effect.
2. **This measures re-ranking, not retrieval.** Every pair here was already
   proposed by the candidate generator. It does not measure whether the correct
   SKU reaches the shortlist at all - that is recall@k, and it is what decides
   whether the embedding belongs beside RapidFuzz rather than beside LightGBM.